In [1]:
import numpy as np
import sys
import os
import time
import random
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

PROJECT_ROOT = "/home/jupyter/project/Student-Camp-Projects"
sys.path.append(PROJECT_ROOT)

from src.models.FNO import ModelFNO1
from src.dataset_utils.dataset import ACTClusterDataset

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)                       
    np.random.seed(seed)                   
    torch.manual_seed(seed)             
    torch.cuda.manual_seed(seed)         
    torch.cuda.manual_seed_all(seed)        

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 128

In [5]:
model = ModelFNO1(depth=4, modes1=24, modes2=24, width=96, hidden_dim=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
loss_fn = torch.nn.MSELoss()

In [6]:
train_indices = np.load("/home/jupyter/project/Student-Camp-Projects/src/data/base/train_indices.npy")
test_indices = np.load("/home/jupyter/project/Student-Camp-Projects/src/data/base/test_indices.npy")

In [7]:
train_dataset = ACTClusterDataset("/home/jupyter/project/Student-Camp-Projects/dataset/", train_indices)
test_dataset = ACTClusterDataset("/home/jupyter/project/Student-Camp-Projects/dataset/", test_indices)

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,     
    pin_memory=True,   
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,     
    pin_memory=True,    
)

In [ ]:
model.train()
losses = []

for epoch in range(EPOCHS):
    start_t = time.time()
    total_loss = 0
    for img, scalars, target in train_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        pred = model(img, scalars)
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss / len(train_loader)) 

    scheduler.step()

    if epoch == 0:
        print("Epoch 1 time:", time.time() - start_t)

    # --- оценка каждые 10 эпох ---
    if (epoch + 1) % 10 == 0:
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for img, scalars, target in test_loader:
                img = img.to(device)
                scalars = scalars.to(device)
                target = target.to(device)

                pred = model(img, scalars)
                preds.append(pred)
                targets.append(target)

        model.train()

        preds = torch.cat(preds)
        targets = torch.cat(targets)

        mae = torch.abs(targets - preds)
        mean_mae = mae.mean().item()
        max_mae = mae.max().item()

        relative_errors = mae / targets
        mean_rel_error = relative_errors.mean().item()
        max_rel_error = relative_errors.max().item()

        print(f"EPOCH={epoch+1}")
        print(f"Средняя MAE {mean_mae:.4f}")
        print(f"Максимальная MAE {max_mae:.4f}")
        print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
        print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")
losses = np.asarray(losses)

EPOCH=70
Средняя MAE 0.1120
Максимальная MAE 1.4959
Средняя относительная ошибка: 0.2063
Максимальная относительная ошибка: 2.7547
EPOCH=80
Средняя MAE 0.1122
Максимальная MAE 1.5042
Средняя относительная ошибка: 0.2108
Максимальная относительная ошибка: 2.7699
EPOCH=100
Средняя MAE 0.1105
Максимальная MAE 1.5105
Средняя относительная ошибка: 0.2037
Максимальная относительная ошибка: 2.7814


In [ ]:
model.eval()  # режим оценки
preds = []
targets = []

with torch.no_grad():
    for img, scalars, target in test_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        pred = model(img, scalars)

        preds.append(pred.cpu())
        targets.append(target.cpu())

preds = torch.cat(preds)
targets = torch.cat(targets)

mae = torch.abs(targets - preds)
mean_mae = mae.mean().item()
max_mae = mae.max().item()

relative_errors = mae / torch.abs(targets)
mean_rel_error = relative_errors.mean().item()
max_rel_error = relative_errors.max().item()

print(f"EPOCH={epoch+1}")
print(f"Средняя MAE: {mean_mae:.4f}")
print(f"Максимальная MAE: {max_mae:.4f}")
print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")

EPOCH=100
Средняя MAE: 0.1105
Максимальная MAE: 1.5105
Средняя относительная ошибка: 0.2037
Максимальная относительная ошибка: 2.7814


In [ ]:
output_dir = "/home/jupyter/project/Student-Camp-Projects/src/results/base/base/"
np.save(output_dir + "base_base_losses.npy", losses)
np.save(output_dir + "base_base_targets.npy", targets)
np.save(output_dir + "base_base_preds.npy", preds)